**ASSIGNMENT 8 One-Hot Encoding; high-cardinality caution**

In [1]:
import pandas as pd

df = pd.DataFrame({
    "Customer_ID": range(101, 113),
    "City": ["Bhopal", "Delhi", "Mumbai", "Bhopal", "Delhi", "Pune",
             "Mumbai", "Pune", "Bhopal", "Delhi", "Pune", "Mumbai"],
    "Payment_Method": ["UPI", "Cash", "Card", "UPI", "Card", "Cash",
                       "UPI", "Card", "Cash", "UPI", "Card", "Cash"],
    "Purchase_Amount": [500,1200,800,1500,700,900,1800,1100,600,1300,950,2000]
})

encoded = pd.get_dummies(df, columns=["City", "Payment_Method"], dtype=int)

print(encoded)
print("Rows:", encoded.shape[0])
print("New columns:", encoded.shape[1] - df.shape[1])



    Customer_ID  Purchase_Amount  City_Bhopal  City_Delhi  City_Mumbai  \
0           101              500            1           0            0   
1           102             1200            0           1            0   
2           103              800            0           0            1   
3           104             1500            1           0            0   
4           105              700            0           1            0   
5           106              900            0           0            0   
6           107             1800            0           0            1   
7           108             1100            0           0            0   
8           109              600            1           0            0   
9           110             1300            0           1            0   
10          111              950            0           0            0   
11          112             2000            0           0            1   

    City_Pune  Payment_Method_Card  P

In [2]:
print("Original shape:", df.shape)
print("Encoded shape:", encoded.shape)

print("Original memory:", df.memory_usage(deep=True).sum())
print("Encoded memory:", encoded.memory_usage(deep=True).sum())


Original shape: (12, 4)
Encoded shape: (12, 9)
Original memory: 1607
Encoded memory: 996


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({
    "Age": [22, 35, 28, 45],
    "City": ["Bhopal", "Delhi", "Mumbai", "Pune"],
    "Browser": ["Chrome", "Firefox", "Safari", "Chrome"],
    "Purchase_Amount": [600, 1500, 900, 2200]
})

X = df[["Age", "City", "Browser"]]

ct = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     ["City", "Browser"])
], remainder="passthrough")

X = ct.fit_transform(X)

print("Shape:", X.shape)
print("Features:", ct.get_feature_names_out())


Shape: (4, 8)
Features: ['cat__City_Bhopal' 'cat__City_Delhi' 'cat__City_Mumbai' 'cat__City_Pune'
 'cat__Browser_Chrome' 'cat__Browser_Firefox' 'cat__Browser_Safari'
 'remainder__Age']


In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = df[["Age", "City", "Browser"]]
y = df["Purchase_Amount"]

model = make_pipeline(ct, LinearRegression())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))


MAE: 44.346013548723704
